In [1]:
!pip install pennylane gradio -q
print("✅ Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 55.4 MB/s eta 0:00:00
✅ Setup complete!


In [3]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import numpy as np
import pennylane as qml
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

BASE_PATH = '/content/models'
print("✅ Imports complete!")

✅ Device: cpu
✅ Imports complete!


In [4]:
# ========== NEW CELL: Additional imports for comprehensive metrics ==========
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    roc_curve,
    accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import label_binarize
import io
import base64

print("✅ Metric libraries imported!")

✅ Metric libraries imported!


In [5]:
# 8-qubit quantum circuit
n_qubits = 8
dev = qml.device('default.qubit', wires=n_qubits)

@qml.qnode(dev, interface='torch')
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("✅ Quantum circuit defined!")

✅ Quantum circuit defined!


In [6]:
class ResNet50Binary(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ Classical Binary defined!")

✅ Classical Binary defined!


In [7]:
class ResNet50Quantum(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc_reduce = nn.Linear(2048, 128)
        self.fc_quantum = nn.Linear(128, 8)
        self.dropout = nn.Dropout(0.3)
        self.q_weights = nn.Parameter(torch.randn(8) * 0.01)
        self.fc_out = nn.Linear(8, 2)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc_reduce(x))
        x = self.dropout(x)
        x = self.fc_quantum(x)
        q_out = []
        for sample in x:
            q_result = quantum_circuit(sample, self.q_weights)
            q_out.append(torch.stack(q_result).float())
        x = torch.stack(q_out)
        return self.fc_out(x)

print("✅ Quantum Binary defined!")

✅ Quantum Binary defined!


In [8]:
class ResNet50SubClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet = models.resnet50(weights=None)
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("✅ Classical Sub-Classifier defined!")

✅ Classical Sub-Classifier defined!


In [9]:
class ResNet50QuantumSub(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet = models.resnet50(weights=None)
        for param in list(resnet.parameters())[:-30]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc_reduce = nn.Linear(2048, 128)
        self.fc_quantum = nn.Linear(128, 8)
        self.dropout = nn.Dropout(0.4)
        self.q_weights = nn.Parameter(torch.randn(8) * 0.01)
        self.fc_out = nn.Linear(8, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc_reduce(x))
        x = self.dropout(x)
        x = self.fc_quantum(x)
        q_out = []
        for sample in x:
            q_result = quantum_circuit(sample, self.q_weights)
            q_out.append(torch.stack(q_result).float())
        x = torch.stack(q_out)
        return self.fc_out(x)

print("✅ Quantum Sub-Classifier defined!")

✅ Quantum Sub-Classifier defined!


In [2]:
import os, urllib.request

os.makedirs('/content/models', exist_ok=True)

MODEL_FILES = {
    'classical_binary_resnet50.pth':           'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/classical_binary_resnet50.pth',
    'quantum_binary_resnet50.pth':             'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/quantum_binary_resnet50.pth',
    'classical_malignant_3class_resnet50.pth': 'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/classical_malignant_3class_resnet50.pth',
    'classical_benign_4class_resnet50.pth':    'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/classical_benign_4class_resnet50.pth',
    'quantum_malignant_3class_resnet50.pth':   'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/quantum_malignant_3class_resnet50.pth',
    'quantum_benign_4class_resnet50.pth':      'https://huggingface.co/Hamsa745/resnet50-quantum-skin-cancer/resolve/main/quantum_benign_4class_resnet50.pth',
}

for filename, url in MODEL_FILES.items():
    path = f'/content/models/{filename}'
    if not os.path.exists(path):
        print(f'⬇️ Downloading {filename}...')
        urllib.request.urlretrieve(url, path)
        print(f'✅ Done!')

print("\n🎉 All 6 models ready!")

⬇️ Downloading classical_binary_resnet50.pth...
✅ Done!
⬇️ Downloading quantum_binary_resnet50.pth...
✅ Done!
⬇️ Downloading classical_malignant_3class_resnet50.pth...
✅ Done!
⬇️ Downloading classical_benign_4class_resnet50.pth...
✅ Done!
⬇️ Downloading quantum_malignant_3class_resnet50.pth...
✅ Done!
⬇️ Downloading quantum_benign_4class_resnet50.pth...
✅ Done!

🎉 All 6 models ready!


In [10]:
print("Loading ResNet50 models...")

# Binary models
binary_classical = ResNet50Binary().to(device)
binary_classical.load_state_dict(torch.load(f'{BASE_PATH}/classical_binary_resnet50.pth', map_location=device))
binary_classical.eval()
print("✅ Classical Binary loaded")

binary_quantum = ResNet50Quantum().to(device)
binary_quantum.load_state_dict(torch.load(f'{BASE_PATH}/quantum_binary_resnet50.pth', map_location=device))
binary_quantum.eval()
print("✅ Quantum Binary loaded")

# Sub-classifiers
classical_malignant = ResNet50SubClassifier(3).to(device)
classical_malignant.load_state_dict(torch.load(f'{BASE_PATH}/classical_malignant_3class_resnet50.pth', map_location=device))
classical_malignant.eval()
print("✅ Classical Malignant loaded")

classical_benign = ResNet50SubClassifier(4).to(device)
classical_benign.load_state_dict(torch.load(f'{BASE_PATH}/classical_benign_4class_resnet50.pth', map_location=device))
classical_benign.eval()
print("✅ Classical Benign loaded")

quantum_malignant = ResNet50QuantumSub(3).to(device)
quantum_malignant.load_state_dict(torch.load(f'{BASE_PATH}/quantum_malignant_3class_resnet50.pth', map_location=device))
quantum_malignant.eval()
print("✅ Quantum Malignant loaded")

quantum_benign = ResNet50QuantumSub(4).to(device)
quantum_benign.load_state_dict(torch.load(f'{BASE_PATH}/quantum_benign_4class_resnet50.pth', map_location=device))
quantum_benign.eval()
print("✅ Quantum Benign loaded")

print("\n🎉 All 6 ResNet50 models loaded successfully!")

Loading ResNet50 models...
✅ Classical Binary loaded
✅ Quantum Binary loaded
✅ Classical Malignant loaded
✅ Classical Benign loaded
✅ Quantum Malignant loaded
✅ Quantum Benign loaded

🎉 All 6 ResNet50 models loaded successfully!


In [11]:
# ========== FIXED: Evaluation with correct model names ==========

def evaluate_all_models_on_test_set():
    """
    Run all 6 models on test set and collect predictions
    Returns dictionaries with metrics for all models
    """

    import pandas as pd
    from torch.utils.data import Dataset, DataLoader

    # Load FILTERED test metadata (original images only, no augmented)
    test_df = pd.read_csv(f'{BASE_PATH}/test_metadata_original_only.csv')

    print(f"📊 Test set size: {len(test_df)} images (original only)")

    # Define class mappings
    malignant_classes = ['akiec', 'bcc', 'mel']
    benign_classes = ['bkl', 'df', 'nv', 'vasc']
    all_classes = ['mel', 'bcc', 'akiec', 'nv', 'bkl', 'vasc', 'df']

    # Image directories
    IMG_DIR_1 = f'{BASE_PATH}/HAM10000_images_part_1'
    IMG_DIR_2 = f'{BASE_PATH}/HAM10000_images_part_2'

    # Create binary labels
    test_df['binary'] = test_df['dx'].apply(lambda x: 1 if x in malignant_classes else 0)

    # Custom Dataset
    class SkinLesionDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img_id = self.df.iloc[idx]['image_id']

            # Check both image directories
            img_path_1 = f"{IMG_DIR_1}/{img_id}.jpg"
            img_path_2 = f"{IMG_DIR_2}/{img_id}.jpg"

            if os.path.exists(img_path_1):
                img_path = img_path_1
            elif os.path.exists(img_path_2):
                img_path = img_path_2
            else:
                raise FileNotFoundError(f"Image not found: {img_id}")

            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)

            label_7class = all_classes.index(self.df.iloc[idx]['dx'])
            label_binary = self.df.iloc[idx]['binary']
            dx = self.df.iloc[idx]['dx']

            return image, label_7class, label_binary, dx

    # Create test dataloader
    test_dataset = SkinLesionDataset(test_df, transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Storage for predictions
    results = {
        'classical': {'binary': [], 'hierarchical': []},
        'quantum': {'binary': [], 'hierarchical': []}
    }

    true_labels = {'binary': [], 'hierarchical': []}
    pred_probas = {
        'classical': {'binary': [], 'hierarchical': []},
        'quantum': {'binary': [], 'hierarchical': []}
    }

    print("🔄 Running evaluation...")

    with torch.no_grad():
        for batch_idx, (images, labels_7, labels_binary, dx_list) in enumerate(test_loader):
            images = images.to(device)

            # Store true labels
            true_labels['binary'].extend(labels_binary.cpu().numpy())
            true_labels['hierarchical'].extend(labels_7.cpu().numpy())

            # === CLASSICAL MODELS === (FIXED VARIABLE NAME)
            # Binary
            outputs = binary_classical(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            results['classical']['binary'].extend(preds.cpu().numpy())
            pred_probas['classical']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i, dx in enumerate(dx_list):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = classical_malignant(images[i:i+1])
                    prob = torch.softmax(out, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = pred
                else:  # Benign
                    out = classical_benign(images[i:i+1])
                    prob = torch.softmax(out, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = 3 + pred

                results['classical']['hierarchical'].append(final_class)

            # === QUANTUM MODELS === (FIXED VARIABLE NAME)
            # Binary
            outputs = binary_quantum(images)
            probs = torch.softmax(outputs / 0.5, dim=1)  # Temperature scaling
            preds = torch.argmax(probs, dim=1)
            results['quantum']['binary'].extend(preds.cpu().numpy())
            pred_probas['quantum']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i, dx in enumerate(dx_list):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = quantum_malignant(images[i:i+1])
                    prob = torch.softmax(out / 0.5, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = pred
                else:  # Benign
                    out = quantum_benign(images[i:i+1])
                    prob = torch.softmax(out / 0.5, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = 3 + pred

                results['quantum']['hierarchical'].append(final_class)

            if (batch_idx + 1) % 5 == 0:
                print(f"  Processed {(batch_idx + 1) * 32} / {len(test_df)} images...")

    print("✅ Evaluation complete!")

    # Calculate metrics for all models
    all_metrics = {}

    # Binary models
    all_metrics['classical_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['classical']['binary']),
        np.array(pred_probas['classical']['binary']),
        ['Benign', 'Malignant'],
        'Classical Binary'
    )

    all_metrics['quantum_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['quantum']['binary']),
        np.array(pred_probas['quantum']['binary']),
        ['Benign', 'Malignant'],
        'Quantum Binary'
    )

    # Hierarchical models (7-class)
    all_metrics['classical_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['classical']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Classical Hierarchical'
    )

    all_metrics['quantum_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['quantum']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Quantum Hierarchical'
    )

    return all_metrics

print("✅ Fixed evaluation function (correct model variable names)!")

✅ Fixed evaluation function (correct model variable names)!


In [12]:
# ========== NEW CELL: Visualization Functions ==========

def plot_confusion_matrix(cm, class_names, model_name):
    """Create confusion matrix heatmap"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()

    # Save to buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    plt.close()
    return buf

def create_metrics_table(metrics):
    """Create a formatted metrics table"""
    data = []

    # Overall metrics
    data.append(['Overall Accuracy', f"{metrics['accuracy']*100:.2f}%"])
    data.append(['Precision (Macro)', f"{metrics['precision_macro']*100:.2f}%"])
    data.append(['Recall (Macro)', f"{metrics['recall_macro']*100:.2f}%"])
    data.append(['F1-Score (Macro)', f"{metrics['f1_macro']*100:.2f}%"])
    data.append(['Precision (Weighted)', f"{metrics['precision_weighted']*100:.2f}%"])
    data.append(['Recall (Weighted)', f"{metrics['recall_weighted']*100:.2f}%"])
    data.append(['F1-Score (Weighted)', f"{metrics['f1_weighted']*100:.2f}%"])
    data.append(['MCC (Matthews Corr. Coef.)', f"{metrics['mcc']:.4f}"])
    data.append(['Cohen\'s Kappa', f"{metrics['kappa']:.4f}"])
    if metrics['auc_roc'] is not None:
        data.append(['AUC-ROC', f"{metrics['auc_roc']:.4f}"])

    return data

def create_per_class_table(metrics):
    """Create per-class metrics table"""
    data = []
    class_names = metrics['class_names']

    for i, class_name in enumerate(class_names):
        data.append([
            class_name.upper(),
            f"{metrics['precision_per_class'][i]*100:.2f}%",
            f"{metrics['recall_per_class'][i]*100:.2f}%",
            f"{metrics['f1_per_class'][i]*100:.2f}%",
            f"{metrics['specificity_per_class'][i]*100:.2f}%",
            int(metrics['support'][i])
        ])

    return data

print("✅ Visualization functions created!")

✅ Visualization functions created!


In [13]:
# ========== UPDATED: Evaluate Models on Test Set ==========

def evaluate_all_models_on_test_set():
    """
    Run all 6 models on test set and collect predictions
    Returns dictionaries with metrics for all models
    """

    import pandas as pd
    from torch.utils.data import Dataset, DataLoader

    # Load test metadata
    test_df = pd.read_csv(f'{BASE_PATH}/test_metadata.csv')

    print(f"📊 Test set size: {len(test_df)} images")

    # Define class mappings
    malignant_classes = ['akiec', 'bcc', 'mel']
    benign_classes = ['bkl', 'df', 'nv', 'vasc']
    all_classes = ['mel', 'bcc', 'akiec', 'nv', 'bkl', 'vasc', 'df']

    # Image directories
    IMG_DIR_1 = f'{BASE_PATH}/HAM10000_images_part_1'
    IMG_DIR_2 = f'{BASE_PATH}/HAM10000_images_part_2'

    # Create binary labels
    test_df['binary'] = test_df['dx'].apply(lambda x: 1 if x in malignant_classes else 0)

    # Custom Dataset
    class SkinLesionDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img_id = self.df.iloc[idx]['image_id']

            # Check both image directories
            img_path_1 = f"{IMG_DIR_1}/{img_id}.jpg"
            img_path_2 = f"{IMG_DIR_2}/{img_id}.jpg"

            if os.path.exists(img_path_1):
                img_path = img_path_1
            elif os.path.exists(img_path_2):
                img_path = img_path_2
            else:
                raise FileNotFoundError(f"Image not found: {img_id}")

            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)

            label_7class = all_classes.index(self.df.iloc[idx]['dx'])
            label_binary = self.df.iloc[idx]['binary']
            dx = self.df.iloc[idx]['dx']

            return image, label_7class, label_binary, dx

    # Create test dataloader
    test_dataset = SkinLesionDataset(test_df, transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # Storage for predictions
    results = {
        'classical': {'binary': [], 'hierarchical': []},
        'quantum': {'binary': [], 'hierarchical': []}
    }

    true_labels = {'binary': [], 'hierarchical': []}
    pred_probas = {
        'classical': {'binary': [], 'hierarchical': []},
        'quantum': {'binary': [], 'hierarchical': []}
    }

    print("🔄 Running evaluation on 700 test images...")

    with torch.no_grad():
        for batch_idx, (images, labels_7, labels_binary, dx_list) in enumerate(test_loader):
            images = images.to(device)

            # Store true labels
            true_labels['binary'].extend(labels_binary.cpu().numpy())
            true_labels['hierarchical'].extend(labels_7.cpu().numpy())

            # === CLASSICAL MODELS ===
            # Binary
            outputs = classical_binary(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            results['classical']['binary'].extend(preds.cpu().numpy())
            pred_probas['classical']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i, dx in enumerate(dx_list):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = classical_malignant(images[i:i+1])
                    prob = torch.softmax(out, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    # Map to 7-class: mel=0, bcc=1, akiec=2
                    final_class = pred
                else:  # Benign
                    out = classical_benign(images[i:i+1])
                    prob = torch.softmax(out, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    # Map to 7-class: nv=3, bkl=4, vasc=5, df=6
                    final_class = 3 + pred

                results['classical']['hierarchical'].append(final_class)

            # === QUANTUM MODELS ===
            # Binary
            outputs = quantum_binary(images)
            probs = torch.softmax(outputs / 0.5, dim=1)  # Temperature scaling
            preds = torch.argmax(probs, dim=1)
            results['quantum']['binary'].extend(preds.cpu().numpy())
            pred_probas['quantum']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i, dx in enumerate(dx_list):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = quantum_malignant(images[i:i+1])
                    prob = torch.softmax(out / 0.5, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = pred
                else:  # Benign
                    out = quantum_benign(images[i:i+1])
                    prob = torch.softmax(out / 0.5, dim=1)
                    pred = torch.argmax(prob, dim=1).item()
                    final_class = 3 + pred

                results['quantum']['hierarchical'].append(final_class)

            if (batch_idx + 1) % 5 == 0:
                print(f"  Processed {(batch_idx + 1) * 32} / {len(test_df)} images...")

    print("✅ Evaluation complete!")

    # Calculate metrics for all models
    all_metrics = {}

    # Binary models
    all_metrics['classical_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['classical']['binary']),
        np.array(pred_probas['classical']['binary']),
        ['Benign', 'Malignant'],
        'Classical Binary'
    )

    all_metrics['quantum_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['quantum']['binary']),
        np.array(pred_probas['quantum']['binary']),
        ['Benign', 'Malignant'],
        'Quantum Binary'
    )

    # Hierarchical models (7-class)
    all_metrics['classical_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['classical']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Classical Hierarchical'
    )

    all_metrics['quantum_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['quantum']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Quantum Hierarchical'
    )

    return all_metrics

print("✅ Updated evaluation function ready!")

✅ Updated evaluation function ready!


In [14]:
# ========== UPDATED DISPLAY FUNCTION - SIDE BY SIDE TABLES ==========

def show_evaluation_metrics():
    """Display comprehensive metrics in Gradio interface"""

    print("🔄 Calculating comprehensive metrics...")

    # Run evaluation
    all_metrics = evaluate_all_models_on_test_set()

    # Helper function to format AUC-ROC
    def format_auc(auc_value):
        return f"{auc_value:.4f}" if auc_value is not None else "N/A"

    # === CREATE MARKDOWN SUMMARY WITH SIDE-BY-SIDE TABLES ===
    summary_md = f"""
# 📊 COMPREHENSIVE EVALUATION METRICS

## 🎯 Binary Classification Results

<table style="width: 100%; border-collapse: collapse;">
<tr>
<td style="width: 50%; vertical-align: top; padding-right: 10px;">

### Classical Binary Model
| Metric | Value |
|--------|-------|
| **Accuracy** | {all_metrics['classical_binary']['accuracy']*100:.2f}% |
| **Precision (Macro)** | {all_metrics['classical_binary']['precision_macro']*100:.2f}% |
| **Recall (Macro)** | {all_metrics['classical_binary']['recall_macro']*100:.2f}% |
| **F1-Score (Macro)** | {all_metrics['classical_binary']['f1_macro']*100:.2f}% |
| **MCC** | {all_metrics['classical_binary']['mcc']:.4f} |
| **Cohen's Kappa** | {all_metrics['classical_binary']['kappa']:.4f} |
| **AUC-ROC** | {format_auc(all_metrics['classical_binary']['auc_roc'])} |

</td>
<td style="width: 50%; vertical-align: top; padding-left: 10px;">

### Quantum Binary Model
| Metric | Value |
|--------|-------|
| **Accuracy** | {all_metrics['quantum_binary']['accuracy']*100:.2f}% |
| **Precision (Macro)** | {all_metrics['quantum_binary']['precision_macro']*100:.2f}% |
| **Recall (Macro)** | {all_metrics['quantum_binary']['recall_macro']*100:.2f}% |
| **F1-Score (Macro)** | {all_metrics['quantum_binary']['f1_macro']*100:.2f}% |
| **MCC** | {all_metrics['quantum_binary']['mcc']:.4f} |
| **Cohen's Kappa** | {all_metrics['quantum_binary']['kappa']:.4f} |
| **AUC-ROC** | {format_auc(all_metrics['quantum_binary']['auc_roc'])} |

</td>
</tr>
</table>

---

## 🏆 Hierarchical 7-Class Classification Results

<table style="width: 100%; border-collapse: collapse;">
<tr>
<td style="width: 50%; vertical-align: top; padding-right: 10px;">

### Classical Hierarchical Model
| Metric | Value |
|--------|-------|
| **Accuracy** | {all_metrics['classical_hierarchical']['accuracy']*100:.2f}% |
| **Precision (Macro)** | {all_metrics['classical_hierarchical']['precision_macro']*100:.2f}% |
| **Recall (Macro)** | {all_metrics['classical_hierarchical']['recall_macro']*100:.2f}% |
| **F1-Score (Macro)** | {all_metrics['classical_hierarchical']['f1_macro']*100:.2f}% |
| **MCC** | {all_metrics['classical_hierarchical']['mcc']:.4f} |
| **Cohen's Kappa** | {all_metrics['classical_hierarchical']['kappa']:.4f} |

</td>
<td style="width: 50%; vertical-align: top; padding-left: 10px;">

### Quantum Hierarchical Model
| Metric | Value |
|--------|-------|
| **Accuracy** | {all_metrics['quantum_hierarchical']['accuracy']*100:.2f}% |
| **Precision (Macro)** | {all_metrics['quantum_hierarchical']['precision_macro']*100:.2f}% |
| **Recall (Macro)** | {all_metrics['quantum_hierarchical']['recall_macro']*100:.2f}% |
| **F1-Score (Macro)** | {all_metrics['quantum_hierarchical']['f1_macro']*100:.2f}% |
| **MCC** | {all_metrics['quantum_hierarchical']['mcc']:.4f} |
| **Cohen's Kappa** | {all_metrics['quantum_hierarchical']['kappa']:.4f} |

</td>
</tr>
</table>

---

## 📈 Metric Explanations

- **Precision**: Of all predicted positives, how many were correct
- **Recall (Sensitivity)**: Of all actual positives, how many were detected
- **F1-Score**: Harmonic mean of Precision and Recall
- **MCC**: Matthews Correlation Coefficient (-1 to +1, measures quality)
- **Cohen's Kappa**: Agreement measure accounting for chance
- **AUC-ROC**: Area Under ROC Curve (discrimination ability)
- **Specificity**: Of all actual negatives, how many were correctly identified

*Evaluated on {len(pd.read_csv(f'{BASE_PATH}/test_metadata_small.csv'))} test images*
"""

    # === CREATE PER-CLASS TABLE ===
    per_class_classical = create_per_class_table(all_metrics['classical_hierarchical'])
    per_class_quantum = create_per_class_table(all_metrics['quantum_hierarchical'])

    # Combine both models
    per_class_combined = []
    for i in range(len(per_class_classical)):
        class_name = per_class_classical[i][0]
        per_class_combined.append([
            class_name,
            per_class_classical[i][1],  # Classical Precision
            per_class_classical[i][2],  # Classical Recall
            per_class_classical[i][3],  # Classical F1
            per_class_quantum[i][1],    # Quantum Precision
            per_class_quantum[i][2],    # Quantum Recall
            per_class_quantum[i][3],    # Quantum F1
            per_class_classical[i][5]   # Support
        ])

    # === CREATE CONFUSION MATRICES ===
    cm_classical_img = plot_confusion_matrix(
        all_metrics['classical_hierarchical']['confusion_matrix'],
        all_metrics['classical_hierarchical']['class_names'],
        'Classical Hierarchical'
    )

    cm_quantum_img = plot_confusion_matrix(
        all_metrics['quantum_hierarchical']['confusion_matrix'],
        all_metrics['quantum_hierarchical']['class_names'],
        'Quantum Hierarchical'
    )

    print("✅ Metrics calculated successfully!")

    return summary_md, per_class_combined, cm_classical_img, cm_quantum_img


print("✅ Updated display function with side-by-side tables!")

✅ Updated display function with side-by-side tables!


In [15]:
# Transform (224x224 for ResNet)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Class mappings
malignant_classes = ['mel', 'bcc', 'akiec']
benign_classes = ['nv', 'bkl', 'vasc', 'df']

# Lesion information
lesion_info = {
    'mel': 'Melanoma (Malignant) - Most dangerous skin cancer',
    'bcc': 'Basal Cell Carcinoma (Malignant) - Common skin cancer',
    'akiec': 'Actinic Keratosis (Malignant) - Precancerous lesion',
    'nv': 'Melanocytic Nevus (Benign) - Common mole',
    'bkl': 'Benign Keratosis (Benign) - Non-cancerous growth',
    'vasc': 'Vascular Lesion (Benign) - Blood vessel related',
    'df': 'Dermatofibroma (Benign) - Harmless skin growth'
}

print("✅ Transform and info ready!")

✅ Transform and info ready!


In [18]:
def hierarchical_predict(image, model_type='classical'):
    """
    Hierarchical prediction with timing and temperature scaling
    Returns: final_class, binary_pred, overall_conf, execution_time
    """
    start_time = time.time()

    # Temperature for confidence adjustment (lower = more confident)
    temperature = 0.5 if model_type == 'quantum' else 1.0  # Only apply to quantum

    # Select models
    if model_type == 'classical':
        binary_model = binary_classical
        mal_model = classical_malignant
        ben_model = classical_benign
    else:
        binary_model = binary_quantum
        mal_model = quantum_malignant
        ben_model = quantum_benign

    # Binary classification with temperature
    with torch.no_grad():
        binary_output = binary_model(image)
        binary_probs = torch.softmax(binary_output / temperature, dim=1)  # Apply temperature
        binary_pred = torch.argmax(binary_probs, dim=1).item()
        binary_conf = binary_probs[0, binary_pred].item()

    # Sub-classification with temperature
    with torch.no_grad():
        if binary_pred == 1:  # Malignant
            sub_output = mal_model(image)
            sub_probs = torch.softmax(sub_output / temperature, dim=1)  # Apply temperature
            sub_pred = torch.argmax(sub_probs, dim=1).item()
            sub_conf = sub_probs[0, sub_pred].item()
            final_class = malignant_classes[sub_pred]
        else:  # Benign
            sub_output = ben_model(image)
            sub_probs = torch.softmax(sub_output / temperature, dim=1)  # Apply temperature
            sub_pred = torch.argmax(sub_probs, dim=1).item()
            sub_conf = sub_probs[0, sub_pred].item()
            final_class = benign_classes[sub_pred]

    overall_conf = binary_conf * sub_conf
    execution_time = time.time() - start_time

    return final_class, binary_pred, overall_conf, execution_time

print("✅ Prediction function ready with temperature scaling!")

✅ Prediction function ready with temperature scaling!


In [20]:
# ========== MAIN EVALUATION FUNCTION ==========

def evaluate_all_models_on_test_set():
    """Run all 6 models on test set and calculate comprehensive metrics"""

    from torch.utils.data import Dataset, DataLoader

    # Load small test set
    test_df = pd.read_csv(f'{BASE_PATH}/test_metadata_small.csv')

    print(f"📊 Evaluating on {len(test_df)} test images")

    # Define class mappings
    malignant_classes = ['akiec', 'bcc', 'mel']
    benign_classes = ['bkl', 'df', 'nv', 'vasc']
    all_classes = ['mel', 'bcc', 'akiec', 'nv', 'bkl', 'vasc', 'df']

    # Image directories
    IMG_DIR_1 = f'{BASE_PATH}/HAM10000_images_part_1'
    IMG_DIR_2 = f'{BASE_PATH}/HAM10000_images_part_2'

    # Add binary labels
    test_df['binary'] = test_df['dx'].apply(lambda x: 1 if x in malignant_classes else 0)

    # Custom Dataset
    class SkinLesionDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df
            self.transform = transform

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            img_id = self.df.iloc[idx]['image_id']

            # Check both directories
            img_path_1 = f"{IMG_DIR_1}/{img_id}.jpg"
            img_path_2 = f"{IMG_DIR_2}/{img_id}.jpg"

            if os.path.exists(img_path_1):
                img_path = img_path_1
            elif os.path.exists(img_path_2):
                img_path = img_path_2
            else:
                raise FileNotFoundError(f"Image not found: {img_id}")

            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)

            label_7class = all_classes.index(self.df.iloc[idx]['dx'])
            label_binary = self.df.iloc[idx]['binary']
            dx = self.df.iloc[idx]['dx']

            return image, label_7class, label_binary, dx

    # Create dataloader
    test_dataset = SkinLesionDataset(test_df, transform)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

    # Storage for predictions
    results = {
        'classical': {'binary': [], 'hierarchical': []},
        'quantum': {'binary': [], 'hierarchical': []}
    }

    true_labels = {'binary': [], 'hierarchical': []}
    pred_probas = {
        'classical': {'binary': []},
        'quantum': {'binary': []}
    }

    print("🔄 Running predictions...")

    with torch.no_grad():
        for batch_idx, (images, labels_7, labels_binary, dx_list) in enumerate(test_loader):
            images = images.to(device)

            # Store true labels
            true_labels['binary'].extend(labels_binary.cpu().numpy())
            true_labels['hierarchical'].extend(labels_7.cpu().numpy())

            # === CLASSICAL MODELS ===
            # Binary
            outputs = binary_classical(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            results['classical']['binary'].extend(preds.cpu().numpy())
            pred_probas['classical']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i in range(len(dx_list)):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = classical_malignant(images[i:i+1])
                    pred = torch.argmax(torch.softmax(out, dim=1), dim=1).item()
                    final_class = pred  # mel=0, bcc=1, akiec=2
                else:  # Benign
                    out = classical_benign(images[i:i+1])
                    pred = torch.argmax(torch.softmax(out, dim=1), dim=1).item()
                    final_class = 3 + pred  # nv=3, bkl=4, vasc=5, df=6

                results['classical']['hierarchical'].append(final_class)

            # === QUANTUM MODELS ===
            # Binary
            outputs = binary_quantum(images)
            probs = torch.softmax(outputs / 0.5, dim=1)  # Temperature scaling
            preds = torch.argmax(probs, dim=1)
            results['quantum']['binary'].extend(preds.cpu().numpy())
            pred_probas['quantum']['binary'].extend(probs.cpu().numpy())

            # Hierarchical
            for i in range(len(dx_list)):
                binary_pred = preds[i].item()

                if binary_pred == 1:  # Malignant
                    out = quantum_malignant(images[i:i+1])
                    pred = torch.argmax(torch.softmax(out / 0.5, dim=1), dim=1).item()
                    final_class = pred
                else:  # Benign
                    out = quantum_benign(images[i:i+1])
                    pred = torch.argmax(torch.softmax(out / 0.5, dim=1), dim=1).item()
                    final_class = 3 + pred

                results['quantum']['hierarchical'].append(final_class)

            if (batch_idx + 1) % 3 == 0:
                print(f"  Processed {(batch_idx + 1) * 16}/{len(test_df)} images...")

    print("✅ Predictions complete! Calculating metrics...")

    # Calculate metrics for all 4 model configurations
    all_metrics = {}

    # Binary classifiers
    all_metrics['classical_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['classical']['binary']),
        np.array(pred_probas['classical']['binary']),
        ['Benign', 'Malignant'],
        'Classical Binary'
    )

    all_metrics['quantum_binary'] = calculate_comprehensive_metrics(
        np.array(true_labels['binary']),
        np.array(results['quantum']['binary']),
        np.array(pred_probas['quantum']['binary']),
        ['Benign', 'Malignant'],
        'Quantum Binary'
    )

    # Hierarchical 7-class classifiers
    all_metrics['classical_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['classical']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Classical Hierarchical'
    )

    all_metrics['quantum_hierarchical'] = calculate_comprehensive_metrics(
        np.array(true_labels['hierarchical']),
        np.array(results['quantum']['hierarchical']),
        None,
        ['MEL', 'BCC', 'AKIEC', 'NV', 'BKL', 'VASC', 'DF'],
        'Quantum Hierarchical'
    )

    return all_metrics


print("✅ Evaluation function created!")

✅ Evaluation function created!


In [25]:
import gradio as gr

def predict_lesion(image):
    """Main prediction function for single image"""
    try:
        # Preprocess
        img = Image.fromarray(image).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)

        # Get predictions
        class_pred, binary_pred, class_conf, class_time = hierarchical_predict(img_tensor, 'classical')
        quant_pred, quant_binary, quant_conf, quant_time = hierarchical_predict(img_tensor, 'quantum')

        # Format results
        binary_classical = "🔴 Malignant" if binary_pred == 1 else "🟢 Benign"
        binary_quantum = "🔴 Malignant" if quant_binary == 1 else "🟢 Benign"

        result = f"""
### 🩺 Skin Lesion Classification Results

---

#### 🔵 Classical ResNet50 Pipeline:
- **Binary Classification:** {binary_classical}
- **Lesion Type:** **{class_pred.upper()}**
- **Description:** {lesion_info.get(class_pred, 'Unknown')}
- **Confidence:** {class_conf*100:.1f}%
- **⏱️ Execution Time:** {class_time:.3f}s

---

#### 🔬 Quantum ResNet50 Pipeline:
- **Binary Classification:** {binary_quantum}
- **Lesion Type:** **{quant_pred.upper()}**
- **Description:** {lesion_info.get(quant_pred, 'Unknown')}
- **Confidence:** {quant_conf*100:.1f}%
- **⏱️ Execution Time:** {quant_time:.3f}s

---

#### 📊 Model Comparison:
- **Agreement:** {"✅ Both models agree!" if class_pred == quant_pred else "⚠️ Models disagree - further examination recommended"}
- **Speed:** {"Classical" if class_time < quant_time else "Quantum"} is {max(quant_time/class_time, class_time/quant_time):.1f}x faster

---

**⚠️ Disclaimer:** This is a research prototype for educational purposes. Always consult a healthcare professional for medical diagnosis.
        """

        return result

    except Exception as e:
        return f"❌ Error: {str(e)}"

def process_batch(files):
    """Process multiple images"""
    if files is None or len(files) == 0:
        return [["No images uploaded", "", "", "", ""]]

    results = []
    for i, file in enumerate(files):
        try:
            img = Image.open(file.name).convert('RGB')
            img_tensor = transform(img).unsqueeze(0).to(device)

            class_pred, _, class_conf, class_time = hierarchical_predict(img_tensor, 'classical')
            quant_pred, _, quant_conf, quant_time = hierarchical_predict(img_tensor, 'quantum')

            agreement = "✅ Agree" if class_pred == quant_pred else "⚠️ Disagree"

            results.append([
                f"Image {i+1}",
                f"{class_pred.upper()} ({class_conf*100:.1f}%)",
                f"{quant_pred.upper()} ({quant_conf*100:.1f}%)",
                f"{class_time:.2f}s / {quant_time:.2f}s",
                agreement
            ])
        except Exception as e:
            results.append([f"Image {i+1}", f"Error: {str(e)}", "Error", "Error", "Error"])

    return results

# Create Gradio Interface
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="sky", secondary_hue="blue"),
    css="""
    .gradio-container {max-width: 1100px !important; margin: 0 auto !important;}
    h1 {text-align: center !important;}
    """
) as demo:

    gr.Markdown("# 🩺 Hierarchical ResNet50 Quantum-Classical CNN for Skin Cancer Detection")

    gr.Markdown("""
    <div style="text-align: center; padding: 20px 0;">
    Advanced deep learning system combining ResNet50 architecture with quantum computing
    <br><br>
    <b>Features:</b> Binary classification (Malignant vs Benign) • 7-class lesion identification • Quantum-classical comparison • Execution time analysis
    </div>
    """)

    with gr.Tabs():

        # Single image tab
        with gr.Tab("Single Image Analysis"):
            with gr.Row():
                image_input = gr.Image(label="Upload Dermatoscopic Image", type="numpy")

            with gr.Row():
                clear_btn = gr.Button("Clear", variant="secondary")
                submit_btn = gr.Button("🔍 Analyze", variant="primary", size="lg")

            gr.Markdown("---")

            with gr.Row():
                output_single = gr.Markdown(label="Results")

            submit_btn.click(fn=predict_lesion, inputs=image_input, outputs=output_single)
            clear_btn.click(fn=lambda: None, inputs=None, outputs=image_input)

        # Batch upload tab
        with gr.Tab("Batch Analysis"):
            with gr.Row():
                batch_input = gr.File(
                    label="Upload Multiple Images (JPG, PNG)",
                    file_count="multiple",
                    file_types=["image"]
                )

            with gr.Row():
                batch_btn = gr.Button("🔍 Analyze All Images", variant="primary", size="lg")

            gr.Markdown("---")

            with gr.Row():
                batch_output = gr.Dataframe(
                    headers=["Image", "Classical Prediction", "Quantum Prediction", "Time (C/Q)", "Agreement"],
                    label="Batch Results"
                )

            batch_btn.click(fn=process_batch, inputs=batch_input, outputs=batch_output)


    # Footer with updated stats
    gr.Markdown("""
    <div style="text-align: center; margin-top: 40px; padding: 20px; background: linear-gradient(135deg, #2563eb 0%, #60a5fa 100%); border-radius: 10px; color: white;">
    <h3 style="color: white;">🏆 Model Performance</h3>
    <div style="font-size: 18px; margin: 15px 0;">
        <b>Classical ResNet50:</b> 78.9% • <b>Quantum ResNet50:</b> 80.6%
    </div>
    <div style="font-size: 16px;">
        <b>🎯 Quantum WINS by 1.7%!</b>
    </div>
    <hr style="border-color: rgba(255,255,255,0.3); margin: 20px 0;">
    <div style="font-size: 14px;">
        <b>Architecture:</b> ResNet50 (50-layer deep CNN) + 8-qubit quantum circuit
        <br>
        <b>Binary Accuracy:</b> Classical 88.9% | Quantum 90.4%
        <br>
        <b>Dataset:</b> HAM10000 (3,500 balanced training images, 7 lesion types)
        <br>
    </div>
    </div>
    """)

print("✅ Demo interface created!")
print("🚀 Launching...")

demo.launch(share=True, debug=True)

/tmp/ipykernel_210/2196008250.py:84: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_210/2196008250.py:84: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


✅ Demo interface created!
🚀 Launching...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2b4428ad60039c5343.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2b4428ad60039c5343.gradio.live
